In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 4: Merge SUMO trajectories with the OMNeT++ feature matrix
# =============================================================================
# Step:         4 of 7
# Summary:      Align SUMO/OMNeT++ vehicle ids and merge the SUMO trajectory with the Step 3 feature matrix.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.1.1
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 4 — Merge SUMO trajectories with the OMNeT++ feature matrix

Part of the **FUMD-AI preprocessing workflow** (step 4 of 7).

**Purpose.** SUMO and OMNeT++ each assign their own vehicle ids, which do not
match each other directly. This notebook uses a SUMO<->OMNeT id mapping file
to align both datasets onto a single id space, then merges the per-timestep
SUMO trajectory (position, speed, lane, ...) with the per-timestep OMNeT++
network-quality matrix produced in Step 3.

**Inputs:**
- `VEHICLES_PATH` - SUMO trajectory CSV (tab-separated), one row per
  vehicle per timestep, including lagged position columns `x-1..x-7` / `y-1..y-7`.
- `OMNET_PATH` - the wide OMNeT++ matrix produced by Step 3.
- `MAPPING_PATH` - SUMO<->VEINS/OMNeT++ id mapping, one line per vehicle,
  formatted as `SUMO=<sumo_id> VEINS=<veins_id>` (e.g. `SUMO=0 VEINS=0`).
  This file is produced directly by the OMNeT++/VEINS simulation itself
  and is treated here as a given input, not something this workflow
  generates.

**Output:** `COMBINED_PATH` - one merged row per `(vehicle, time)` present
in both datasets.


In [ ]:
import pandas as pd


In [ ]:
# ---- Parameters ----
# Defaults chain onto Step 2's/Step 3's default outputs and the bundled
# example mapping file, so this notebook runs out of the box.
VEHICLES_PATH = "sumo_trajectory.csv"          # Step 2 output (input)
OMNET_PATH = "omnet_feature_matrix.csv"        # Step 3 output (input)
MAPPING_PATH = "example-data/sumo_veins_mapping.txt"  # SUMO<->VEINS/OMNeT id mapping (input)
COMBINED_PATH = "combined_dataset.csv"         # merged output -> feeds Step 5's INPUT_PATH


## 1. Load all three inputs

In [ ]:
# SUMO trajectory: one row per (vehicle, timestep), tab-separated
vehicles_data = pd.read_csv(VEHICLES_PATH, delimiter="\t")
# Step 3 output: one row per (vehicle, timestep), tab-separated
omnet_data = pd.read_csv(OMNET_PATH, delimiter="\t")

# id mapping: one "SUMO=<id> VEINS=<id>" line per vehicle, no header row.
# Split each whitespace-separated token on "=" to pull out the two integer
# ids (VEINS is the OMNeT++ vehicular-networking framework, so its id here
# is the same "omnet" id used everywhere else in this workflow).
cars_data = pd.read_csv(MAPPING_PATH, sep=r"\s+", header=None, names=["sumo_tok", "omnet_tok"])
cars_data["sumo"] = cars_data["sumo_tok"].str.split("=").str[1].astype(int)
cars_data["omnet"] = cars_data["omnet_tok"].str.split("=").str[1].astype(int)
cars_data = cars_data[["omnet", "sumo"]]

print("SUMO vehicles:", vehicles_data["veh_id"].nunique())
print("OMNeT vehicles:", omnet_data["Object"].nunique())
print("mapping rows:", len(cars_data))
vehicles_data.head()


## 2. Resolve duplicate SUMO ids in the mapping

Occasionally the same SUMO vehicle id gets reused by two (or more) different
OMNeT++ vehicles across a run (SUMO recycles ids once a vehicle leaves the
simulation). When that happens we keep the **first** OMNeT id seen for that
SUMO id and remap every later duplicate OMNeT id onto it, so every SUMO id
maps to exactly one OMNeT id.

On a run with no id reuse this cell is a no-op.

**Note on ids reused more than twice.** An earlier version of this cell
built the remapping from two independently-deduplicated `.unique()` arrays
(`omnet_last` -> `omnet_first`) and paired them up positionally. That only
works when every reused SUMO id is reused *exactly once* (a group of size
2). On a larger, busier real run (`VoipDl-Urban-900_1`, 900+ vehicles) some
SUMO ids get reused twice (groups of size 3), which made the two arrays
come out different lengths after deduplication (`ValueError: Replacement
lists must match in length`), and would have silently mis-paired ids even
in cases where the lengths happened to match by coincidence. The cell now
builds an explicit `{old_omnet_id: kept_omnet_id}` dictionary instead,
which maps every later occurrence straight to the first one seen for its
SUMO id regardless of how many times that SUMO id was reused.


In [ ]:
# find every row whose "sumo" id appears more than once in the mapping,
# in original file order within each SUMO id (kind="stable" guarantees
# that, unlike the default sort).
dup_groups = cars_data[cars_data["sumo"].duplicated(keep=False)].sort_values("sumo", kind="stable")

# For each duplicated SUMO id, keep the first OMNeT id seen and map every
# later OMNeT id for that same SUMO id onto it. Built as an explicit
# {old_id: kept_id} dict (not a pair of .unique() arrays - see markdown
# above) so this is correct regardless of how many times a SUMO id repeats.
first_omnet_by_sumo = {}
remap = {}
for sumo_id, omnet_id in zip(dup_groups["sumo"], dup_groups["omnet"]):
    kept = first_omnet_by_sumo.setdefault(sumo_id, omnet_id)
    if omnet_id != kept:
        remap[omnet_id] = kept

print(f"{len(dup_groups)} duplicate-SUMO-id rows found ({len(remap)} to remap)")

# rewrite the later OMNeT ids (in both the feature matrix and the mapping
# table itself) to point at the OMNeT id we decided to keep
omnet_data["Object"] = omnet_data["Object"].replace(remap)
cars_data["omnet"] = cars_data["omnet"].replace(remap)
# now every SUMO id maps to exactly one OMNeT id - drop the now-redundant duplicate rows
cars_data = cars_data.drop_duplicates(keep="first", ignore_index=True)


## 3. Remap the SUMO trajectory ids into OMNeT id space

After this, `vehicles_data["veh_id"]` and `omnet_data["Object"]` refer to
the same vehicle using the same id, so they can be merged directly.


In [ ]:
# cars_data is a clean 1:1 mapping at this point, so the i-th value of
# "sumo".unique() and the i-th value of "omnet".unique() are guaranteed to
# be the pairing for the same underlying row/vehicle.
sumo_ids = cars_data["sumo"].unique()
omnet_ids = cars_data["omnet"].unique()
vehicles_data["veh_id"] = vehicles_data["veh_id"].replace(sumo_ids, omnet_ids)

ids_now_aligned = set(vehicles_data["veh_id"].unique()) <= set(omnet_data["Object"].unique())
print("every remapped SUMO veh_id has a matching OMNeT Object id:", ids_now_aligned)


## 4. Merge on (time, vehicle id) and save

In [ ]:
# how="right" keeps every OMNeT row (the network-quality signal we care
# about) and attaches the matching SUMO trajectory row where one exists.
combined_data = pd.merge(
    vehicles_data, omnet_data,
    left_on=["t", "veh_id"], right_on=["Time", "Object"],
    how="right",
)
print("rows before dropping unmatched:", len(combined_data))

# rows with no matching SUMO sample (different observation windows between
# the two simulators) end up all-NaN on the SUMO side - drop them
combined_data = combined_data.dropna()
print("rows after dropping unmatched:", len(combined_data))

combined_data.to_csv(COMBINED_PATH, index=False)
print(f"saved {COMBINED_PATH} - shape {combined_data.shape}")
combined_data.head()
